# 04 - Difference Maps and Profiles

This notebook computes and visualizes difference maps between FUS LCD variants and wild-type.

## Contents
1. Compute difference maps (Variant - WT)
2. Visualize difference map heatmaps
3. Extract difference profiles
4. Identify regions of largest perturbation
5. Quantify mutation effects

In [ ]:
# Add src to path
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Import modules
from src.sequences import VARIANTS, get_variant
from src.intermaps import (
    compute_all_intermaps,
    InterMapConfig,
    compute_difference_map,
    compute_all_difference_maps,
    compute_map_statistics,
    compute_mean_interaction_profile,
)
from src.plotting import (
    plot_difference_map,
    plot_difference_maps_panel,
    plot_interaction_map,
    plot_profiles_comparison,
    set_publication_style,
    save_figure,
)

In [ ]:
set_publication_style()

In [ ]:
# Compute intermaps (unnormalized for physical interpretation)
config = InterMapConfig(smooth=True, sigma=2.0, normalize=False)
intermaps = compute_all_intermaps(VARIANTS, config=config)
print(f"Computed {len(intermaps)} interaction maps")

## 1. Compute Difference Maps

Difference maps show how mutations perturb the interaction landscape:
- **Positive values (green)**: Variant has weaker (less negative) interactions → loss of attraction
- **Negative values (pink)**: Variant has stronger (more negative) interactions → gain of attraction

In [ ]:
# Compute all difference maps relative to WT
diff_maps = compute_all_difference_maps(intermaps, reference_name='WT')

print("Difference Map Statistics (Variant - WT):")
print("=" * 60)

for name, dmap in diff_maps.items():
    stats = compute_map_statistics(dmap)
    print(f"\n{name}:")
    print(f"  Mean: {stats['mean']:+.4f} kT")
    print(f"  Std:  {stats['std']:.4f} kT")
    print(f"  Range: [{stats['min']:.4f}, {stats['max']:.4f}] kT")
    print(f"  Total perturbation: {stats['total_interaction']:.2f} kT")

## 2. Visualize Difference Maps

In [ ]:
# Single difference map: AllY_to_S - WT
fig = plot_difference_map(
    diff_maps['AllY_to_S'],
    title='Difference Map: AllY_to_S − WT\n(Green = loss of attraction)',
    figsize=(8, 7)
)
plt.show()

In [ ]:
# All difference maps panel
fig = plot_difference_maps_panel(
    diff_maps,
    reference_name='WT',
    ncols=2,
    figsize=(12, 12)
)
plt.suptitle('Difference Maps: Variant − Wild-Type', fontsize=14, y=1.02)
plt.show()

In [ ]:
# Compare key variants side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

variants_to_show = ['AllY_to_S', 'AllY_to_A', 'AllY_to_F']
titles = ['Y→S (polar)', 'Y→A (hydrophobic)', 'Y→F (aromatic)']

# Find common scale
all_vals = np.concatenate([diff_maps[v].ravel() for v in variants_to_show])
abs_max = max(abs(all_vals.min()), abs(all_vals.max()))

for ax, variant, title in zip(axes, variants_to_show, titles):
    dmap = diff_maps[variant]
    im = ax.imshow(dmap, cmap='PiYG', vmin=-abs_max, vmax=abs_max, aspect='equal')
    ax.set_title(f'{variant}\n({title})')
    ax.set_xlabel('Residue')
    ax.set_ylabel('Residue')

# Shared colorbar
fig.subplots_adjust(right=0.85)
cbar_ax = fig.add_axes([0.88, 0.15, 0.03, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('ΔEnergy (kT)')

plt.suptitle('Comparison of Mutation Effects on Interaction Landscape', fontsize=14)
plt.tight_layout(rect=[0, 0, 0.85, 1])
plt.show()

## 3. Difference Profiles

1D profiles showing perturbation along the sequence.

In [ ]:
# Compute difference profiles
wt_profile = compute_mean_interaction_profile(intermaps['WT'])
diff_profiles = {}

for name, dmap in diff_maps.items():
    diff_profiles[name] = compute_mean_interaction_profile(dmap)

print("Difference Profile Magnitudes:")
for name, profile in diff_profiles.items():
    print(f"  {name}: max|Δ| = {np.max(np.abs(profile)):.4f} kT")

In [ ]:
# Visualize difference profiles
fig, ax = plt.subplots(figsize=(14, 5))

x = np.arange(len(wt_profile)) + 1
colors = plt.cm.tab10(np.linspace(0, 1, len(diff_profiles)))

for (name, profile), color in zip(diff_profiles.items(), colors):
    ax.plot(x, profile, label=name, color=color, linewidth=1.5)

ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.fill_between(x, 0, diff_profiles['AllY_to_S'], alpha=0.2, color='green', label='_nolegend_')

# Mark tyrosine positions
wt = get_variant('WT')
tyr_pos = wt.tyrosine_positions
for pos in tyr_pos:
    ax.axvline(x=pos+1, color='orange', alpha=0.2, linewidth=0.5)

ax.set_xlabel('Residue Position')
ax.set_ylabel('ΔMean Interaction Energy (kT)')
ax.set_title('Difference Profiles (Variant − WT)\nOrange lines = Tyrosine positions in WT')
ax.legend(loc='upper right')
ax.set_xlim(1, len(wt_profile))

plt.tight_layout()
plt.show()

## 4. Identify Regions of Largest Perturbation

In [ ]:
# Find top perturbed positions
def find_top_perturbations(diff_profile, n_top=10):
    """Find positions with largest perturbation."""
    abs_profile = np.abs(diff_profile)
    top_indices = np.argsort(abs_profile)[-n_top:][::-1]
    return [(i+1, diff_profile[i]) for i in top_indices]  # 1-indexed

print("Top 10 Perturbed Positions (AllY_to_S):")
print("=" * 40)
top_positions = find_top_perturbations(diff_profiles['AllY_to_S'], n_top=10)

wt_seq = get_variant('WT').sequence
for pos, delta in top_positions:
    aa = wt_seq[pos-1]
    print(f"  Position {pos:3d} ({aa}): Δ = {delta:+.4f} kT")

In [ ]:
# Heat map of perturbation intensity
fig, ax = plt.subplots(figsize=(14, 3))

# Stack profiles as rows
profile_matrix = np.vstack([diff_profiles[name] for name in diff_profiles])
names = list(diff_profiles.keys())

abs_max = np.max(np.abs(profile_matrix))
im = ax.imshow(profile_matrix, cmap='PiYG', aspect='auto', vmin=-abs_max, vmax=abs_max)

ax.set_yticks(np.arange(len(names)))
ax.set_yticklabels(names)
ax.set_xlabel('Residue Position')
ax.set_title('Perturbation Heatmap: All Variants vs WT')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('ΔEnergy (kT)')

plt.tight_layout()
plt.show()

## 5. Quantify Mutation Effects

In [ ]:
# Summary statistics
def compute_perturbation_metrics(diff_map):
    """Compute summary metrics for a difference map."""
    return {
        'mean_delta': np.mean(diff_map),
        'total_delta': np.sum(diff_map) / 2,  # Divide by 2 for symmetric
        'rmsd': np.sqrt(np.mean(diff_map**2)),
        'max_loss': np.max(diff_map),  # Max positive = loss of attraction
        'max_gain': np.min(diff_map),  # Max negative = gain of attraction
        'frac_perturbed': np.mean(np.abs(diff_map) > 0.01),
    }

print("Perturbation Metrics Summary:")
print("=" * 70)
print(f"{'Variant':12s} {'Mean Δ':>10s} {'Total Δ':>10s} {'RMSD':>10s} {'Max Loss':>10s}")
print("-" * 70)

for name, dmap in diff_maps.items():
    metrics = compute_perturbation_metrics(dmap)
    print(f"{name:12s} {metrics['mean_delta']:+10.4f} {metrics['total_delta']:+10.2f} "
          f"{metrics['rmsd']:10.4f} {metrics['max_loss']:+10.4f}")

In [ ]:
# Bar plot comparison
metrics_dict = {name: compute_perturbation_metrics(dmap) for name, dmap in diff_maps.items()}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

names = list(metrics_dict.keys())
x = np.arange(len(names))

# Total perturbation
total_delta = [metrics_dict[n]['total_delta'] for n in names]
axes[0].bar(x, total_delta, color='#E74C3C', edgecolor='black')
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=45, ha='right')
axes[0].set_ylabel('Total ΔEnergy (kT)')
axes[0].set_title('Total Perturbation')
axes[0].axhline(y=0, color='black', linewidth=0.5)

# RMSD
rmsd = [metrics_dict[n]['rmsd'] for n in names]
axes[1].bar(x, rmsd, color='#3498DB', edgecolor='black')
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, rotation=45, ha='right')
axes[1].set_ylabel('RMSD (kT)')
axes[1].set_title('Root Mean Square Deviation')

# Max loss
max_loss = [metrics_dict[n]['max_loss'] for n in names]
axes[2].bar(x, max_loss, color='#2ECC71', edgecolor='black')
axes[2].set_xticks(x)
axes[2].set_xticklabels(names, rotation=45, ha='right')
axes[2].set_ylabel('Max Loss of Attraction (kT)')
axes[2].set_title('Maximum Local Perturbation')

plt.suptitle('Quantification of Mutation Effects', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Save difference maps
output_path = Path("../data/outputs/difference_maps.npz")
np.savez_compressed(output_path, **diff_maps)
print(f"Difference maps saved to {output_path}")

## Summary

In this notebook we:
1. Computed difference maps showing perturbations relative to WT
2. Visualized difference maps as heatmaps
3. Extracted 1D difference profiles along the sequence
4. Identified positions with largest perturbation
5. Quantified mutation effects (total perturbation, RMSD, max loss)

**Key observations**:
- AllY_to_S shows the largest total perturbation (complete loss of Y-Y attractions)
- AllY_to_A has similar but slightly smaller perturbation
- AllY_to_F has minimal perturbation (F preserves aromatic character)
- Perturbation is concentrated at original tyrosine positions

**Next**: Notebook 05 - Sticker-Linker Metrics